# Run every lesson on Apple silicon (MPS)

Runs every lesson's pipeline on a Mac with an M-series chip, training the networks on its GPU through PyTorch's Metal (MPS) backend, then reconciles every model to its paper and times MPS against the CPU. Open it in a local clone (see *Local* in the README: `uv venv`, `uv pip install -e ".[dev]"`, then `jupyter lab`). Elsewhere it falls back to the default device.

**Expect MPS to be no faster:** these networks are small, so GPU kernel-launch overhead outweighs the speedup (section 4 measures it on your machine). The full reference run fits 15 seeds per frequency network and takes hours; set `ENV = "smoke"` below for a two-minute pass on 2% of the data. To log to Weights & Biases as well, set `WANDB_PROJECT` (and `WANDB_API_KEY`) before starting Jupyter; `scripts/full_run.sh` is the same full run from the terminal.

In [ ]:
# Setup: runs anywhere Jupyter runs. Opened on its own (Colab, Kaggle, SageMaker, Azure ML,
# any hosted Jupyter), this cell clones and installs the repo; in a clone or the container it
# uses that checkout.
import os, sys, importlib.util
# Colab must run on a GPU runtime: stop here otherwise (set DAS_ALLOW_CPU=1 to override).
if "google.colab" in sys.modules and os.environ.get("DAS_ALLOW_CPU") != "1":
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "This notebook needs a GPU runtime. In Colab: Runtime > Change runtime type > "
            "T4 GPU > Save, then Runtime > Run all. (To run on CPU anyway: "
            "import os; os.environ['DAS_ALLOW_CPU'] = '1')"
        )
    print("GPU:", torch.cuda.get_device_name(0))

REPO_DIR = next((d for d in (os.getcwd(), os.path.dirname(os.getcwd()))
                 if os.path.isdir(os.path.join(d, "src", "deep_actuarial_science"))), None)
if REPO_DIR is None:
    if sys.version_info < (3, 10):
        raise RuntimeError("Needs Python 3.10+: choose a newer kernel or image")
    # a writable, persistent home: Colab, Kaggle, SageMaker notebook instances, else ~
    base = next(d for d in ("/content", "/kaggle/working", os.path.expanduser("~/SageMaker"),
                            os.path.expanduser("~")) if os.path.isdir(d) and os.access(d, os.W_OK))
    REPO_DIR = os.path.join(base, "deep-actuarial-science")
    if not os.path.exists(REPO_DIR):  # Kaggle: turn on Internet in the notebook settings
        !git clone -q https://github.com/actuarialsociety/deep-actuarial-science {REPO_DIR}
    if importlib.util.find_spec("deep_actuarial_science") is None:
        !pip install -q -e {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, "src")
os.environ.setdefault("DO_NOT_TRACK", "1")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

from deep_actuarial_science.notebook import (run, load, show, leaderboard, reconciliation, findings,
                                             parameters, mlflow_ui, device_summary)
device_summary()
FAST = os.environ.get("DAS_ENV") == "smoke"   # CI runs every notebook in smoke mode


## 1. Train on the Apple GPU

In [ ]:
import time, torch
ENV = "smoke" if FAST else "local"   # "smoke": every pipeline on 2% of the data
MPS = torch.backends.mps.is_available()
if MPS:
    os.environ["DAS_DEVICE"] = "mps"   # read by every network at training time
else:
    print("No MPS (not an Apple-silicon Mac, or PyTorch without Metal): using the default device.")
device_summary()

## 2. Run every pipeline
Frequency (GLMs, LightGBM, FNN, CANN, LocalGLMnet), mortality (Lee–Carter and the deep models) and reserving (chain-ladder, neural ODP), each tracked in MLflow. LightGBM trains single-threaded on macOS, because its OpenMP runtime and PyTorch's crash in one process.

In [ ]:
timings = {}
for p in ("frequency", "mortality", "reserving"):
    t0 = time.perf_counter()
    run(p, env=ENV)
    timings[p] = time.perf_counter() - t0
    print(f"{p}: {timings[p] / 60:.1f} min")

## 3. Every model against its paper
`paper` is the published value, `reference_run` this repo's frozen full run (`reference/`), `this_run` what you just ran. **PASS**: within the tolerance of the paper. A reduced run (smoke, `scale` < 1) reports SKIPPED.

In [ ]:
import pandas as pd
PIPELINES = ('frequency', 'mortality', 'reserving')
pd.concat(reconciliation(p, env=ENV) for p in PIPELINES)

## 4. MPS against the CPU
The same FNN (one seed, 10% of the data) trained on each device. GPU and CPU results agree to about 0.002 Poisson deviance.

In [ ]:
small = {} if FAST else {"frequency.data.scale": 0.1, "frequency.models.fnn.n_runs": 1}
rows = []
for dev in (["cpu", "mps"] if MPS else ["cpu"]):
    os.environ["DAS_DEVICE"] = dev
    t0 = time.perf_counter()
    run("frequency", models=["fnn"], params=small, env=ENV)
    m = load("fnn.freq_metrics", env=ENV)
    rows.append({"device": dev, "wall_seconds": time.perf_counter() - t0, **m})
os.environ["DAS_DEVICE"] = "mps" if MPS else "cpu"
# fit_seconds is training alone; wall_seconds adds data preparation and logging
pd.DataFrame(rows).set_index("device")[["fit_seconds", "wall_seconds", "deviance_test", "gini_test", "balance_test"]]

## Explore the runs in MLflow
Parameters, metrics, figures and the benchmark table of every run above. MLflow runs on the notebook's machine: in Colab it opens below and in a new tab through Colab's proxy; in SageMaker and Azure ML through Jupyter's proxy; on Kaggle, which exposes no ports, query the runs with `mlflow.search_runs` instead.

In [ ]:
mlflow_ui()